In [ ]:
import os
import uvicorn
import requests
import nest_asyncio
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from openai import OpenAI
from dotenv import load_dotenv
import database

# 환경 변수 로드
load_dotenv()

# Jupyter 환경에서 FastAPI 실행을 위한 설정
nest_asyncio.apply()

app = FastAPI()

# CORS 설정: 모든 허용
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def getModelResponse(userPrompt):
    """ 설정된 모델(GPT 또는 OLLAMA)에 따라 응답을 생성합니다. """
    modelType = os.getenv("MODEL_TYPE", "GPT")
    
    if modelType == "GPT":
        # GPT 모델 연동
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": userPrompt}]
        )
        return response.choices[0].message.content
        
    elif modelType == "OLLAMA":
        # Ollama 모델 연동
        ollamaUrl = f"{os.getenv('OLLAMA_BASE_URL')}/api/generate"
        payload = {
            "model": os.getenv("OLLAMA_MODEL", "gemma4:e2b"),
            "prompt": userPrompt,
            "stream": False
        }
        response = requests.post(ollamaUrl, json=payload)
        return response.json().get("response", "")
        
    else:
        return "지원하지 않는 모델 타입입니다."

@app.post("/chat")
async def chatEndpoint(request: Request):
    """ 채팅 API 엔드포인트입니다. """
    try:
        body = await request.json()
        userMessage = body.get("message", "")
        
        # 모델 응답 생성
        aiResponse = getModelResponse(userMessage)
        
        # 결과 반환
        return {"success": True, "data": aiResponse}
        
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"success": False, "message": str(e)}
        )

@app.get("/history")
async def getHistory():
    """ 데이터베이스에서 채팅 기록을 가져옵니다. """
    try:
        query = "SELECT * FROM chat_history ORDER BY createdAt DESC"
        historyList = database.fetchAll(query)
        
        return {"success": True, "data": historyList}
        
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"success": False, "message": str(e)}
        )

if __name__ == "__main__":
    # 서버 실행 (Port: 8000)
    print("Starting FastAPI Server...")
    uvicorn.run(app, host="0.0.0.0", port=8000)